# Notebook 05: Supervised Learning — Logistic Regression

## Credit Card Customer Churn & Segmentation
**Objective:** Demonstrate baseline model evaluation, train Logistic Regression with balanced class weighting, perform Stratified 5-Fold Cross-Validation, optimize the classification threshold, compute business lift, and interpret odds ratios.

---
### The Accuracy Paradox
With a 16.07% churn rate, a naive dummy model predicting 'Existing Customer' for everyone yields **83.96% accuracy** but detects **0% of churners**. We explicitly demonstrate why accuracy is a flawed metric for imbalanced problems.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import json

from src.data.load_data import load_raw_data
from src.data.preprocess import split_data
from src.models.train_churn_model import train_and_evaluate_churn_model
from src.utils.helpers import MODELS_DIR

meta = train_and_evaluate_churn_model()
print("Model Training & Evaluation Completed.")

## 1. Model Performance: Baseline vs Logistic Regression

In [ ]:
comparison = pd.DataFrame([
    {
        "Model": "Dummy Classifier (Baseline)",
        "Threshold": 0.50,
        "Accuracy": meta["baseline_metrics"]["accuracy"],
        "Precision": meta["baseline_metrics"]["precision"],
        "Recall": meta["baseline_metrics"]["recall"],
        "F1 Score": meta["baseline_metrics"]["f1"],
        "ROC-AUC": meta["baseline_metrics"]["roc_auc"],
    },
    {
        "Model": "Balanced Logistic Regression (Default)",
        "Threshold": meta["default_threshold"],
        "Accuracy": meta["metrics_default_threshold"]["accuracy"],
        "Precision": meta["metrics_default_threshold"]["precision"],
        "Recall": meta["metrics_default_threshold"]["recall"],
        "F1 Score": meta["metrics_default_threshold"]["f1"],
        "ROC-AUC": meta["metrics_default_threshold"]["roc_auc"],
    },
    {
        "Model": "Balanced Logistic Regression (Optimized)",
        "Threshold": meta["selected_threshold"],
        "Accuracy": meta["metrics_optimized_threshold"]["accuracy"],
        "Precision": meta["metrics_optimized_threshold"]["precision"],
        "Recall": meta["metrics_optimized_threshold"]["recall"],
        "F1 Score": meta["metrics_optimized_threshold"]["f1"],
        "ROC-AUC": meta["metrics_optimized_threshold"]["roc_auc"],
    }
])
comparison

## 2. ROC & Precision-Recall Curves
Logistic Regression achieves an impressive **ROC-AUC of 0.934** and **PR-AUC of 0.763** on the holdout test set.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC Curve
roc_data = pd.DataFrame(meta["curves"]["roc_curve"])
axes[0].plot(roc_data["fpr"], roc_data["tpr"], label=f"LogReg (AUC = {meta['metrics_default_threshold']['roc_auc']:.3f})", color='#1e3a8a', lw=2)
axes[0].plot([0, 1], [0, 1], 'k--', label="Random Guess")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate (Recall)")
axes[0].set_title("ROC Curve")
axes[0].legend()

# PR Curve
pr_data = pd.DataFrame(meta["curves"]["pr_curve"])
axes[1].plot(pr_data["recall"], pr_data["precision"], label=f"LogReg (PR-AUC = {meta['metrics_default_threshold']['pr_auc']:.3f})", color='#059669', lw=2)
axes[1].axhline(0.1604, color='r', linestyle='--', label="Baseline Churn Rate (16.0%)")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Threshold Analysis & Business Lift
In banking, deciding which customers to contact requires balancing outreach budget against churn loss.
- At the **top 10% risk decile**, precision is **84.16%** with a **5.25x Lift**, capturing **52.3%** of all churners in just 10% of customer accounts.

In [ ]:
sweep_df = pd.DataFrame(meta["threshold_sweep"])
plt.figure(figsize=(10, 5))
plt.plot(sweep_df["threshold"], sweep_df["precision"], label="Precision", color='#2563eb')
plt.plot(sweep_df["threshold"], sweep_df["recall"], label="Recall", color='#dc2626')
plt.plot(sweep_df["threshold"], sweep_df["f1"], label="F1 Score", color='#16a34a', lw=2)
plt.axvline(meta["selected_threshold"], color='black', linestyle=':', label=f"Optimal F1 Threshold ({meta['selected_threshold']})")
plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Precision, Recall, and F1 across Classification Thresholds")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Odds Ratio Interpretation
Each unit increase in a standardized feature multiplies the odds of churn by $e^{\beta}$.

In [ ]:
top_drivers = pd.DataFrame(meta["all_coefficients"])
print("Top 5 Features Increasing Churn Risk:")
display(top_drivers[top_drivers["coefficient"] > 0].head(5))

print("\nTop 5 Features Protecting Retention:")
display(top_drivers[top_drivers["coefficient"] < 0].head(5))